<a href="https://colab.research.google.com/github/jerperkins/zhuang-dialect-analysis/blob/main/CSC575ZhuangProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install Praat-parselmouth
!pip install praat-parselmouth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 79.1 MB/s eta 0:00:00


In [ ]:
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [6]:
from google.colab import drive
import pandas as pd
import ipywidgets as widgets
import librosa
import librosa.display
import os
# Import praat-parselmouth to use Praat's pitch tracking algorithm
# Needs installing once via !pip install praat-parselmouth (using the above cell)
import parselmouth
import matplotlib.pyplot as plt
import numpy as np
# UI that allows filtering, displaying sounds nicely
from IPython.display import display, clear_output, Audio

# 1. Mount Drive (Force remount to see the new shortcut)
drive.mount('/content/drive', force_remount=True)

# 2. Define the Path
PROJECT_PATH = '/content/drive/MyDrive/Zhuang'
METADATA_PATH = os.path.join(PROJECT_PATH, 'FileInfoForRIPA.csv')

# 3. specific check
if os.path.exists(PROJECT_PATH):
    print(f"✅ Found the Zhuang folder at: {PROJECT_PATH}")

    if os.path.exists(METADATA_PATH):
        df = pd.read_csv(METADATA_PATH)
        print("✅ Metadata loaded!")

        # ── Derive extra columns from Filename ────────────────────────────────
        # Dialect  = first 2 characters  (e.g. "WZ" or "DZ")
        # Speaker  = first 4 characters  (e.g. "WZ01")
        # WordID   = token between 2nd and 3rd underscore  (e.g. "12" or "119")
        df['Dialect'] = df['Filename'].astype(str).str[:2]
        df['Speaker'] = df['Filename'].astype(str).str[:4]
        df['WordID']  = df['Filename'].astype(str).str.split('_').str[2]

        # Drop entries the speaker skipped — no recording exists for these
        n_before = len(df)
        df = df[df['Exclude'].astype(str).str.strip().str.lower() != 'skipped'].reset_index(drop=True)
        print(f"ℹ️  Removed {n_before - len(df)} skipped entries. {len(df)} usable rows remain.")

        display(df.head())
    else:
        print(f"❌ Folder found, but could not find 'FileInfoForRIPA.csv' inside it.")
        print("Check if the file is still uploading?")
else:
    print(f"❌ Still cannot find the folder at: {PROJECT_PATH}")
    print("Did you create the shortcut in 'My Drive'?")

# ***********************************************

AUDIO_PATH = os.path.join(PROJECT_PATH, 'Audio')

# Pre-build a set of filenames in the Audio folder — one network listing
# instead of a separate os.path.exists() call per file later.
try:
    _audio_files = set(os.listdir(AUDIO_PATH))
    print(f"ℹ️  Audio index built: {len(_audio_files)} items found.")
except Exception as e:
    _audio_files = set()
    print(f"⚠️  Could not list Audio folder: {e}")

# ── Speaker median f0 cache (computed lazily on first use per speaker) ────────
speaker_f0_cache = {}

def get_speaker_median_f0(speaker_id, n_samples=30):
    """Return median voiced f0 (Hz) for a speaker, sampling up to n_samples files."""
    if speaker_id in speaker_f0_cache:
        return speaker_f0_cache[speaker_id]

    spk_files = df[df['Speaker'] == speaker_id]['Filename'].tolist()
    if not spk_files:
        return None

    import random
    sample = random.sample(spk_files, min(n_samples, len(spk_files)))

    all_f0 = []
    for fname in sample:
        raw = str(fname).strip()
        for suffix in ('', '_E', '_E2S', '_2S'):
            wav_path = os.path.join(AUDIO_PATH, raw + suffix + '.wav')
            if os.path.exists(wav_path):
                try:
                    snd  = parselmouth.Sound(wav_path)
                    vals = snd.to_pitch().selected_array['frequency']
                    all_f0.extend(vals[vals > 0].tolist())
                except Exception:
                    pass
                break

    if not all_f0:
        return None

    median_f0 = float(np.median(all_f0))
    speaker_f0_cache[speaker_id] = median_f0
    return median_f0

# ── Shared widget style ──────────────────────────────────────────────────────
_dd_layout = widgets.Layout(width='160px')

# ── Helper: build a sorted dropdown including 'All' ─────────────────────────
def _dd(col, description, layout=_dd_layout, numeric_sort=False):
    vals = df[col].dropna().unique().tolist()
    if numeric_sort:
        def _sort_key(v):
            try: return (0, int(v))
            except (ValueError, TypeError): return (1, str(v))
        vals = sorted(vals, key=_sort_key)
    else:
        vals = sorted(vals)
    opts = ['All'] + vals
    return widgets.Dropdown(options=opts, value='All',
                            description=description, layout=layout)

# ══════════════════════════════════════════════════════════════════════════════
# LEFT COLUMN FILTERS  (also used by single-column "Plot Audio")
# ══════════════════════════════════════════════════════════════════════════════
tone_dropdown         = _dd('ReclassifiedTone',        'Tone:')
vowel_length_dropdown = _dd('ReclassifiedVowelLength',  'Vowel Len:')
coda_consonant_dropdown = _dd('ReclassifiedCoda',      'Coda:')
onset_dropdown        = _dd('Onset',                   'Onset:')
exclude_dropdown      = _dd('Exclude',                  'Exclude:')
dict_match_dropdown   = _dd('DictionaryMatch',          'Dict Match:')
dialect_dropdown      = _dd('Dialect',                  'Dialect:')
speaker_dropdown      = _dd('Speaker',                  'Speaker:')
word_id_dropdown      = _dd('WordID',                   'Word ID:',  numeric_sort=True)
ons_type_dropdown     = _dd('OnsType',                  'Ons Type:')

# ══════════════════════════════════════════════════════════════════════════════
# RIGHT COLUMN FILTERS  (used only for side-by-side Compare)
# ══════════════════════════════════════════════════════════════════════════════
tone_R         = _dd('ReclassifiedTone',        'Tone:')
vowel_length_R = _dd('ReclassifiedVowelLength',  'Vowel Len:')
coda_R         = _dd('ReclassifiedCoda',         'Coda:')
onset_R        = _dd('Onset',                    'Onset:')
exclude_R      = _dd('Exclude',                   'Exclude:')
dict_match_R   = _dd('DictionaryMatch',           'Dict Match:')
dialect_R      = _dd('Dialect',                   'Dialect:')
speaker_R      = _dd('Speaker',                   'Speaker:')
word_id_R      = _dd('WordID',                    'Word ID:',  numeric_sort=True)
ons_type_R     = _dd('OnsType',                   'Ons Type:')

# ── Global options ────────────────────────────────────────────────────────────
n_input = widgets.BoundedIntText(
    value=10, min=1, max=50, step=1,
    description='Num Files:',
    layout=widgets.Layout(width='150px')
)
spectrogram_dropdown = widgets.Dropdown(
    options=['y', 'n'], value='y',
    description='Show Spec:',
    layout=widgets.Layout(width='150px')
)
random_checkbox = widgets.Checkbox(
    value=False, description='Random?',
    indent=False, layout=widgets.Layout(width='120px')
)
normalize_f0_checkbox = widgets.Checkbox(
    value=False, description='Norm f0?',
    indent=False, layout=widgets.Layout(width='120px')
)

# ── Buttons ───────────────────────────────────────────────────────────────────
search_button  = widgets.Button(description="Plot Audio", button_style='success')
compare_button = widgets.Button(description="Compare",    button_style='info')

output_box   = widgets.Output()
output_left  = widgets.Output(layout=widgets.Layout(width='50%'))
output_right = widgets.Output(layout=widgets.Layout(width='50%'))

# ── Helper: apply all filters to df ──────────────────────────────────────────
def apply_filters(tone, vowel_length, coda, onset, exclude_status,
                  dict_match, dialect, speaker, word_id, ons_type):
    fdf = df.copy()
    if tone         != 'All': fdf = fdf[fdf['ReclassifiedTone']       == tone]
    if vowel_length != 'All': fdf = fdf[fdf['ReclassifiedVowelLength'] == vowel_length]
    if coda         != 'All': fdf = fdf[fdf['ReclassifiedCoda']        == coda]
    if onset        != 'All': fdf = fdf[fdf['Onset']                  == onset]
    if exclude_status != 'All': fdf = fdf[fdf['Exclude']               == exclude_status]
    if dict_match   != 'All': fdf = fdf[fdf['DictionaryMatch']         == dict_match]
    if dialect      != 'All': fdf = fdf[fdf['Dialect']                 == dialect]
    if speaker      != 'All': fdf = fdf[fdf['Speaker']                 == speaker]
    if word_id      != 'All': fdf = fdf[fdf['WordID']                  == word_id]
    if ons_type     != 'All': fdf = fdf[fdf['OnsType']                 == ons_type]
    return fdf

# ── Helper: render one audio file ─────────────────────────────────────────────
def render_file(row, show_spectrogram, ax_wave, ax_spec=None, normalize_f0=False):
    """Draw waveform + f0 line (and optionally spectrogram) for one audio row."""
    raw_name   = str(row['Filename']).strip()
    # Don't double-add .wav if the CSV already stores the extension
    if raw_name.lower().endswith('.wav'):
        file_name = raw_name
    else:
        file_name = raw_name + '.wav'
    audio_path = os.path.join(AUDIO_PATH, file_name)

    if file_name not in _audio_files:
        # Fallback: try _E and _E2S variants (error-marked files where CSV omits the suffix)
        for suffix in ('_E', '_E2S', '_2S'):
            alt_name = raw_name + suffix + '.wav'
            if alt_name in _audio_files:
                file_name = alt_name
                audio_path = os.path.join(AUDIO_PATH, file_name)
                break
        else:
            ax_wave.set_title(f"Missing: {file_name}", color='red')
            print(f"⚠️  Could not find: {file_name} (also tried _E / _E2S variants)")
            return None, None, None
    else:
        audio_path = os.path.join(AUDIO_PATH, file_name)

    # Single load via parselmouth — extract waveform from it to avoid
    # downloading the file twice (previously: librosa.load + parselmouth.Sound)
    snd          = parselmouth.Sound(audio_path)
    y            = snd.values[0].astype(np.float32)   # mono waveform
    sr           = int(snd.sampling_frequency)
    pitch        = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency'].copy()
    pitch_times  = pitch.xs()

    # Spectrogram data
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)

    # Panel 1 – waveform
    librosa.display.waveshow(y, sr=sr, ax=ax_wave)
    ax_wave.set(
        title=(f"{file_name}  |  Tone: {row['ReclassifiedTone']}"
               f"  |  VL: {row['ReclassifiedVowelLength']}"),
        ylabel='Amplitude'
    )

    # F0 overlay as a LINE
    pitch_values[pitch_values == 0] = np.nan   # hide voiceless segments
    ax_pitch = ax_wave.twinx()

    if normalize_f0:
        median_hz = get_speaker_median_f0(str(row['Speaker']))
        if median_hz and median_hz > 0:
            with np.errstate(divide='ignore', invalid='ignore'):
                pitch_plot = np.where(
                    np.isfinite(pitch_values),
                    1200.0 * np.log2(pitch_values / median_hz),
                    np.nan
                )
            ax_pitch.plot(pitch_times, pitch_plot, '-', linewidth=1.5, color='red')
            ax_pitch.set_ylabel('Pitch (cents, rel. speaker median)', color='red')
            ax_pitch.set_ylim(-1200, 1200)
            ax_pitch.axhline(0, color='red', linestyle='--', linewidth=0.6, alpha=0.5)
        else:
            # Fallback to Hz if median unavailable
            ax_pitch.plot(pitch_times, pitch_values, '-', linewidth=1.5, color='red')
            ax_pitch.set_ylabel('Pitch (Hz) [norm unavailable]', color='red')
            ax_pitch.set_ylim(50, 500)
    else:
        ax_pitch.plot(pitch_times, pitch_values, '-', linewidth=1.5, color='red')
        ax_pitch.set_ylabel('Pitch ($f_0$)', color='red')
        ax_pitch.set_ylim(50, 500)

    ax_pitch.tick_params(axis='y', colors='red')

    # Panel 2 – spectrogram (optional)
    if ax_spec is not None:
        librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax_spec)
        ax_spec.set(title='Spectrogram', ylabel='Hz')

    return y, sr, file_name

# ── Single-column plot (Plot Audio button) ────────────────────────────────────
def plot_audio_files(b):
    # Clear compare panes so only single-column view is visible
    with output_left:
        clear_output(wait=True)
    with output_right:
        clear_output(wait=True)
    with output_box:
        clear_output(wait=True)

        filtered_df = apply_filters(
            tone_dropdown.value, vowel_length_dropdown.value,
            coda_consonant_dropdown.value, onset_dropdown.value,
            exclude_dropdown.value, dict_match_dropdown.value,
            dialect_dropdown.value, speaker_dropdown.value,
            word_id_dropdown.value, ons_type_dropdown.value
        )
        n          = n_input.value
        show_spec  = spectrogram_dropdown.value
        use_random = random_checkbox.value

        if use_random:
            display_df = filtered_df.sample(min(n, len(filtered_df)), random_state=None)
            print(f"Found {len(filtered_df)} matching files. Displaying {len(display_df)} randomly selected...")
        else:
            display_df = filtered_df.head(n)
            print(f"Found {len(filtered_df)} matching files. Displaying the first {n}...")

        print("Please wait, generating acoustic data...\n")

        for _, row in display_df.iterrows():
            if show_spec == 'y':
                fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 6), sharex=True)
                ax_wave, ax_spec = ax[0], ax[1]
            else:
                fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 3))
                ax_wave, ax_spec = ax, None

            y, sr, _ = render_file(row, show_spec, ax_wave, ax_spec,
                                    normalize_f0=normalize_f0_checkbox.value)
            plt.tight_layout()
            plt.show()
            if y is not None:
                display(Audio(data=y, rate=sr))

# ── Side-by-side comparison (Compare button) ──────────────────────────────────
def plot_compare(b):
    # Clear single-column pane so only compare view is visible
    with output_box:
        clear_output(wait=True)
    with output_left:
        clear_output(wait=True)
    with output_right:
        clear_output(wait=True)

    n          = n_input.value
    show_spec  = spectrogram_dropdown.value
    use_random = random_checkbox.value

    left_df  = apply_filters(
        tone_dropdown.value, vowel_length_dropdown.value,
        coda_consonant_dropdown.value, onset_dropdown.value,
        exclude_dropdown.value, dict_match_dropdown.value,
        dialect_dropdown.value, speaker_dropdown.value,
        word_id_dropdown.value, ons_type_dropdown.value
    )
    right_df = apply_filters(
        tone_R.value, vowel_length_R.value,
        coda_R.value, onset_R.value,
        exclude_R.value, dict_match_R.value,
        dialect_R.value, speaker_R.value,
        word_id_R.value, ons_type_R.value
    )

    if use_random:
        left_disp  = left_df.sample(min(n, len(left_df)),   random_state=None)
        right_disp = right_df.sample(min(n, len(right_df)), random_state=None)
    else:
        left_disp  = left_df.head(n)
        right_disp = right_df.head(n)

    def _render_col(output_widget, disp_df, label):
        with output_widget:
            print(f"{label}: {len(disp_df)} file(s)")
            for _, row in disp_df.iterrows():
                if show_spec == 'y':
                    fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(6, 5), sharex=True)
                    ax_wave, ax_spec = ax[0], ax[1]
                else:
                    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 3))
                    ax_wave, ax_spec = ax, None
                y, sr, _ = render_file(row, show_spec, ax_wave, ax_spec,
                                        normalize_f0=normalize_f0_checkbox.value)
                plt.tight_layout()
                plt.show()
                if y is not None:
                    display(Audio(data=y, rate=sr))

    _render_col(output_left,  left_disp,  "LEFT")
    _render_col(output_right, right_disp, "RIGHT")

# ── Wire up buttons ───────────────────────────────────────────────────────────
search_button.on_click(plot_audio_files)
compare_button.on_click(plot_compare)

# ── Build UI layout ───────────────────────────────────────────────────────────
_h = lambda text: widgets.HTML(f"<b>{text}</b>")

ui = widgets.VBox([
    _h("Left Column Filters"),
    widgets.HBox([tone_dropdown, vowel_length_dropdown, coda_consonant_dropdown,
                  onset_dropdown, exclude_dropdown]),
    widgets.HBox([dict_match_dropdown, dialect_dropdown, speaker_dropdown,
                  word_id_dropdown, ons_type_dropdown]),
    _h("Right Column Filters (Compare only)"),
    widgets.HBox([tone_R, vowel_length_R, coda_R, onset_R, exclude_R]),
    widgets.HBox([dict_match_R, dialect_R, speaker_R, word_id_R, ons_type_R]),
    _h("Options"),
    widgets.HBox([n_input, spectrogram_dropdown, random_checkbox, normalize_f0_checkbox]),
    widgets.HBox([search_button, compare_button]),
])

display(ui)
display(output_box)                                  # single-column output
display(widgets.HBox([output_left, output_right]))   # side-by-side output

Mounted at /content/drive
✅ Found the Zhuang folder at: /content/drive/MyDrive/Zhuang
✅ Metadata loaded!
ℹ️  Removed 979 skipped entries. 13443 usable rows remain.


,Filename,OriginalIPA,ActualTone,VowelLength,IPA,Coda,ReclassifiedTone,ReclassifiedVowelLength,VowelQuality,PhonemicIPA,ReclassifiedIPA,ReclassifiedCoda,Exclude,DictionaryMatch,Onset,OnsType,Dialect,Speaker,WordID
0,WZ03_0_193_1,fɯːt8,0,long,fɯːt8,obstruent,0,long,ɯə,fɯːət,fɯːt,obstruent,Keep,yes,f,voiceless fricative,WZ,WZ03,193
1,WZ03_3_74_1,poːn3,3,long,poːn3,nasal,3,long,o,poːn,pɔːn,nasal,Keep,yes,p,voiceless stop,WZ,WZ03,74
2,WZ03_5_131_1,puən5_or_tiŋ2,5,short,puən5,nasal,5,long,uə,puːən,puːən,nasal,Keep,yes,p,voiceless stop,WZ,WZ03,131
3,WZ03_1_9_1,ɣon1,1,short,ɣon1,nasal,1,short,o,ɣon,hon,nasal,Keep,yes,h,voiceless fricative,WZ,WZ03,9
4,WZ03_9_179_1,tiːt7,9,long,tiːt7,obstruent,7,short,i,tit,tiət,obstruent,Keep,no,t,voiceless stop,WZ,WZ03,179


ℹ️  Audio index built: 25070 items found.


Output()

In [ ]:
# Load the actual filenames from disk into a set (fast O(1) lookup)
actual_files = set(os.listdir('/content/drive/MyDrive/Zhuang/Audio'))
wav_files = {f for f in actual_files if f.lower().endswith('.wav')}
print(f"WAV files visible to Colab: {len(wav_files)}")
print("Sample WAV filenames:", list(wav_files)[:5])

# Now check every CSV row against the actual files
missing = []
for _, row in df.iterrows():
    raw   = str(row['Filename']).strip()
    fname = raw if raw.lower().endswith('.wav') else raw + '.wav'
    if fname not in actual_files:
        missing.append((row['Filename'], fname))

print(f"\n{len(missing)} CSV entries whose .wav is not found:")
for orig, constructed in missing[:20]:
    print(f"  CSV value: '{orig}'  →  looking for: '{constructed}'")


WAV files visible to Colab: 12534
Sample WAV filenames: ['WZ09_3_58_4.wav', 'WZ13_9_174_3.wav', 'WZ03_2_32_3.wav', 'WZ03_6_127_3.wav', 'DZ02_X_106_5.wav']

4352 CSV entries whose .wav is not found:
  CSV value: 'WZ03_9_182_1'  →  looking for: 'WZ03_9_182_1.wav'
  CSV value: 'WZ03_6_116_1'  →  looking for: 'WZ03_6_116_1.wav'
  CSV value: 'WZ03_7_135_1'  →  looking for: 'WZ03_7_135_1.wav'
  CSV value: 'WZ03_2_42_1'  →  looking for: 'WZ03_2_42_1.wav'
  CSV value: 'WZ03_8_161_1'  →  looking for: 'WZ03_8_161_1.wav'
  CSV value: 'WZ03_0_191_1'  →  looking for: 'WZ03_0_191_1.wav'
  CSV value: 'WZ03_7_142_1'  →  looking for: 'WZ03_7_142_1.wav'
  CSV value: 'WZ03_1_50_1'  →  looking for: 'WZ03_1_50_1.wav'
  CSV value: 'WZ03_0_195_1'  →  looking for: 'WZ03_0_195_1.wav'
  CSV value: 'WZ03_NA_196_1'  →  looking for: 'WZ03_NA_196_1.wav'
  CSV value: 'WZ03_9_176_1'  →  looking for: 'WZ03_9_176_1.wav'
  CSV value: 'WZ03_8_161_2'  →  looking for: 'WZ03_8_161_2.wav'
  CSV value: 'WZ03_9_173_1'  →  look